In [1]:
import ipywidgets.widgets as widgets
import time
import threading
import inspect
import ctypes

In [2]:
from Arm_Lib import Arm_Device
joints_0 = [90, 164, 18, 0, 90, 90]
Arm = Arm_Device()
def arm_move_6(p, s_time = 500):
    for i in range(6):
        id = i + 1
        Arm.Arm_serial_servo_write(id, p[i], s_time)
        time.sleep(.01)
    time.sleep(s_time/1000)
arm_move_6(joints_0)
time.sleep(1)
#Arm.Arm_serial_servo_write6_array(joints_0, 500)


In [3]:
def _async_raise(tid, exctype):
    """raises the exception, performs cleanup if needed"""
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)

In [4]:
import cv2
import numpy as np
from PIL import Image

# 初始化控件
origin_widget = widgets.Image(format='jpeg', width=320, height=240)
mask_widget = widgets.Image(format='jpeg', width=320, height=240)
result_widget = widgets.Image(format='jpeg', width=320, height=240)

# 创建图像容器
image_container = widgets.HBox([origin_widget, mask_widget, result_widget])
#display(image_container)

# 初始化摄像头
cap = cv2.VideoCapture(0) #根据实际video编号替换！
cap.set(3, 640)
cap.set(4, 480)
cap.set(5, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M', 'J', 'P', 'G'))

# 颜色范围
color_lower = np.array([0, 0, 0])
color_upper = np.array([130, 255, 255])

def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

def get_color(img):
    H = []
    color_name = {'name': 'unknown'}
    resized_img = None
    
    if img is None or img.size == 0:
        return resized_img, color_name
    
    try:
        resized_img = cv2.resize(img, (640, 480), interpolation=cv2.INTER_AREA)
        HSV = cv2.cvtColor(resized_img, cv2.COLOR_BGR2HSV)
        cv2.rectangle(resized_img, (280, 180), (360, 260), (0, 255, 0), 2)
        
        for i in range(280, 360):
            for j in range(180, 260):
                H.append(HSV[j, i][0])
        
        if not H:
            return resized_img, color_name
        
        H_min = min(H)
        H_max = max(H)
        
        if H_min >= 0 and H_max <= 10: color_name['name'] = 'red'
        elif H_min >= 21 and H_max <= 34: color_name['name'] = 'yellow'
        elif H_min >= 35 and H_max <= 99: color_name['name'] = 'green'
        elif H_min >= 100 and H_max <= 124: color_name['name'] = 'blue'
    except Exception as e:
        print(f"Error: {e}")
    
    return resized_img, color_name

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1100) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1
[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


In [5]:
def Color_Recognize():
    global color_lower, color_upper
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame, color_name = get_color(frame)
            if frame is None:
                time.sleep(0.01)
                continue
            
            if 'name' in color_name and color_name['name'] != 'unknown':
                if color_name['name'] == 'yellow':
                    color_lower = np.array([21, 98, 110])
                    color_upper = np.array([34, 255, 255])
                elif color_name['name'] == 'red':
                    color_lower = np.array([0, 137, 0])
                    color_upper = np.array([10, 255, 255])
                elif color_name['name'] == 'green':
                    color_lower = np.array([35, 43, 46])
                    color_upper = np.array([99, 255, 255])
                elif color_name['name'] == 'blue':
                    color_lower = np.array([100, 43, 46])
                    color_upper = np.array([124, 255, 255])

            origin_widget.value = bgr8_to_jpeg(frame)
            
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            mask = cv2.inRange(hsv, color_lower, color_upper)
            mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
            mask_widget.value = bgr8_to_jpeg(mask_bgr)
            
            res = cv2.bitwise_and(frame, frame, mask=mask)
            result_widget.value = bgr8_to_jpeg(res)
            
            time.sleep(0.01)
    
    except Exception as e:
        print(f"程序异常退出：{e}")
    
    finally:
        print("释放摄像头资源...")
        cap.release()

In [6]:
# 启动进程
thread1 = threading.Thread(target=Color_Recognize)
thread1.setDaemon=True
thread1.start()
display(image_container)

In [ ]:
# 停止进程
try:
    while True:
        time.sleep(.000001)
except KeyboardInterrupt:
    print("Program closed!")
    stop_thread(thread1)
    if Arm is not None:
        del Arm
    cap.release()